In [5]:
import requests
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# 1. Reliable Ensembl 3'UTR extractor using cDNA sequence + UTR genomic coordinates
def fetch_true_3utr(gene_symbol):
    r1 = requests.get(
        f"https://rest.ensembl.org/lookup/symbol/homo_sapiens/{gene_symbol}?expand=1;content-type=application/json",
        timeout=10
    )
    if r1.status_code != 200:
        return ""
    data = r1.json()
    strand = data.get("strand", 1)
    transcripts = [t for t in data.get("Transcript", []) if t.get("biotype") == "protein_coding"]
    canonical = [t for t in transcripts if t.get("is_canonical") == 1]
    target_t = canonical[0] if canonical else max(transcripts, key=lambda x: x.get("length", 0))
    
    # Fetch full spliced cDNA & Translation coordinates
    enst = target_t["id"]
    r_cdna = requests.get(f"https://rest.ensembl.org/sequence/id/{enst}?type=cdna;content-type=text/plain", timeout=10)
    r_cds  = requests.get(f"https://rest.ensembl.org/sequence/id/{enst}?type=cds;content-type=text/plain", timeout=10)
    if r_cdna.status_code != 200 or r_cds.status_code != 200:
        return ""
    
    cdna_seq = r_cdna.text.strip().upper()
    cds_seq = r_cds.text.strip().upper()
    
    # 3'UTR is everything in the spliced cDNA strictly downstream of the CDS
    cds_end_idx = cdna_seq.find(cds_seq)
    if cds_end_idx != -1:
        return cdna_seq[cds_end_idx + len(cds_seq):]
    return ""

utr_seqs = {g: fetch_true_3utr(g) for g in ["HIF3A", "GAS1", "SCARA5", "LIFR"]}
for g, seq in utr_seqs.items():
    print(f"{g} true spliced 3'UTR length: {len(seq)} bp")

# 2. Scan circLIFR (7,248 bp) & true 3'UTRs for data-driven miRNA seed matches
data_driven_seeds = {
    "hsa-miR-205-5p (MIR205HG)":        "ATGAAGG",
    "hsa-miR-200b/c/429 (MIR200CHG)":   "CAGTATT",
    "hsa-miR-200a/141 (MIR200A)":       "CAGTGTT",
    "hsa-miR-210-3p (MIR210HG)":        "ACGCACA",
    "hsa-miR-34a-5p (MIR34AHG)":        "CACTGCC",
    "hsa-miR-17-5p/20a (MIR17HG)":      "GCACTTT",
    "hsa-miR-145-5p (MIR145)":          "AACTGGA",
    "hsa-miR-1-3p (MIR1-1HG)":          "ACATTCC"
}

print("\n=== Verified Spliced 3'UTR MRE Binding Triangle (circLIFR <--> Partner 3'UTRs) ===")
rows = []
for mir, seed_7mer in data_driven_seeds.items():
    seed_8mer = seed_7mer + "A"
    c_7 = circ_lifr_seq.count(seed_7mer)
    c_8 = circ_lifr_seq.count(seed_8mer)
    hif_7 = utr_seqs["HIF3A"].count(seed_7mer)
    gas_7 = utr_seqs["GAS1"].count(seed_7mer)
    sca_7 = utr_seqs["SCARA5"].count(seed_7mer)
    
    if c_7 > 0 and (hif_7 + gas_7 + sca_7) > 0:
        rows.append({
            "Data_Driven_miRNA": mir,
            "circLIFR_7mer": c_7,
            "circLIFR_8mer": c_8,
            "HIF3A_3UTR": hif_7,
            "GAS1_3UTR": gas_7,
            "SCARA5_3UTR": sca_7
        })

print(pd.DataFrame(rows).to_string(index=False) if rows else "No shared 7mer sites found.")

# 3. Empirical Permutation Test (10,000 random background transcripts) for SensCorr = +0.1346
np.random.seed(42)
bg_pool = mrna_log2.loc[(mrna_log2 > 1.0).sum(axis=1) >= 20]
rand_genes = np.random.choice(bg_pool.index, size=10000, replace=True)

null_sens = []
for rg in rand_genes:
    z_vec = bg_pool.loc[rg].values
    r_cz, _ = spearmanr(c_vec, z_vec)
    r_tz, _ = spearmanr(t_vec, z_vec)
    denom = np.sqrt(1 - r_cz**2) * np.sqrt(1 - r_tz**2)
    if denom > 1e-6:
        r_ct_z = (r_ct - r_cz * r_tz) / denom
        null_sens.append(r_ct - r_ct_z)

null_sens = np.array(null_sens)
emp_pval = np.mean(null_sens >= sens_corr)
print(f"\n=== Empirical Permutation Significance for MIR205HG Mediation ===")
print(f"Observed SensCorr (circLIFR -- MIR205HG -- HIF3A) : +{sens_corr:.4f}")
print(f"Null Mean ± SD across 10,000 random transcripts   : {null_sens.mean():+.4f} ± {null_sens.std():.4f}")
print(f"Empirical Permutation p-value                     : {emp_pval:.4e}")

HIF3A true spliced 3'UTR length: 3811 bp
GAS1 true spliced 3'UTR length: 1380 bp
SCARA5 true spliced 3'UTR length: 2051 bp
LIFR true spliced 3'UTR length: 6927 bp

=== Verified Spliced 3'UTR MRE Binding Triangle (circLIFR <--> Partner 3'UTRs) ===
         Data_Driven_miRNA  circLIFR_7mer  circLIFR_8mer  HIF3A_3UTR  GAS1_3UTR  SCARA5_3UTR
hsa-miR-200a/141 (MIR200A)              1              0           1          1            0
 hsa-miR-34a-5p (MIR34AHG)              1              0           0          1            0
   hsa-miR-145-5p (MIR145)              1              1           0          0            1

=== Empirical Permutation Significance for MIR205HG Mediation ===
Observed SensCorr (circLIFR -- MIR205HG -- HIF3A) : +0.1346
Null Mean ± SD across 10,000 random transcripts   : +0.0140 ± 0.0378
Empirical Permutation p-value                     : 2.1700e-02
